In [28]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [32]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

NUM_CLASSES = len(train_ds.class_names)
CLASS_NAMES = train_ds.class_names

print("Number of classes:", NUM_CLASSES)

Found 128252 files belonging to 245 classes.
Found 42749 files belonging to 245 classes.
Number of classes: 245


In [34]:
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [35]:
import tensorflow as tf
from tensorflow.keras import layers, models

## STEP 1: Loading the trained MobileNet model

In [36]:
baseline_model = tf.keras.models.load_model("model_initial_training.h5")

## STEP 2: Extracting the MobileNetV2 backbone

In [37]:
base_model = baseline_model.layers[0]  # MobileNetV2
base_model.trainable = False  # keep frozen for confirmation

## TEP 3: Defining the Attention Block (SE Block)
### Novelty

In [38]:
def se_block(input_tensor, reduction=16):
    channels = input_tensor.shape[-1]

    x = layers.GlobalAveragePooling2D()(input_tensor)
    x = layers.Dense(channels // reduction, activation="relu")(x)
    x = layers.Dense(channels, activation="sigmoid")(x)
    x = layers.Reshape((1, 1, channels))(x)

    return layers.Multiply()([input_tensor, x])

### STEP 4: Building DAM-Net (Dual-Branch Architecture)
4.1 Input + shared features

In [39]:
# input_layer = layers.Input(shape=(224, 224, 3))
# features = base_model(input_layer)
input_layer = layers.Input(shape=(224,224,3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)
features = base_model(x)

4.2 Branch A — Global Feature Branch

In [40]:
global_branch = layers.GlobalAveragePooling2D()(features)
global_branch = layers.Dense(256, activation="relu")(global_branch)

4.3 Branch B — Attention-Enhanced Local Features

In [41]:
attention_features = se_block(features)
attention_branch = layers.GlobalAveragePooling2D()(attention_features)
attention_branch = layers.Dense(256, activation="relu")(attention_branch)

4.4 Feature Fusion

In [42]:
fusion = layers.Concatenate()([global_branch, attention_branch])

4.5 Classification Head

In [43]:
fusion = layers.Dense(256, activation="relu")(fusion)
fusion = layers.Dropout(0.4)(fusion)

output_layer = layers.Dense(NUM_CLASSES, activation="softmax")(fusion)

### STEP 5: Creating DAM-Net Model

In [44]:
dam_net = models.Model(
    inputs=input_layer,
    outputs=output_layer,
    name="DAM_Net"
)

### STEP 6: Compiling DAM-Net (NEW optimizer is mandatory)

In [45]:
dam_net.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

dam_net.summary()

Model: "DAM_Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (TrueDivide)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract (Subtract) │ (None, 224, 224,  │          0 │ true_divide[0][0] │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ subtract[0][0]    │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 80)        │    102,480 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1280)      │    103,680 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 1,      │          0 │ dense_12[0][0]    │
│                     │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 7, 7,      │          0 │ mobilenetv2_1.00… │
│ (Multiply)          │ 1280)             │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ multiply_1[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ dense_10[0][0],   │
│ (Concatenate)       │                   │            │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 256)       │    131,328 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 245)       │     62,965 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,314,309 (12.64 MB)

 Trainable params: 1,056,325 (4.03 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

### STEP 7: Training DAM-Net

In [46]:
history_dam = dam_net.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 615s 76ms/step - accuracy: 0.1470 - loss: 4.3471 - val_accuracy: 0.4974 - val_loss: 2.6794
Epoch 2/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 768s 96ms/step - accuracy: 0.4540 - loss: 2.2546 - val_accuracy: 0.7430 - val_loss: 1.3817
Epoch 3/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 965s 120ms/step - accuracy: 0.6172 - loss: 1.4341 - val_accuracy: 0.8270 - val_loss: 0.9010
Epoch 4/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 924s 115ms/step - accuracy: 0.7128 - loss: 1.0341 - val_accuracy: 0.8676 - val_loss: 0.6658
Epoch 5/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 614s 77ms/step - accuracy: 0.7707 - loss: 0.8055 - val_accuracy: 0.8874 - val_loss: 0.5276
Epoch 6/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 614s 77ms/step - accuracy: 0.8100 - loss: 0.6519 - val_accuracy: 0.9032 - val_loss: 0.4355
Epoch 7/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 611s 76ms/step - accuracy: 0.8415 - loss: 0.5423 - val_accuracy: 0.9123 - val_loss: 0.3749
Epoch 8/10
8016/8016 ━━━━━━━━━━━━━━━━━━━━ 613s 76ms/step - accuracy

In [47]:
dam_net.save("DAM_Net_fruits360.h5")

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model("model_initial_training.h5")

In [51]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
baseline_loss, baseline_acc = model.evaluate(val_ds)
dam_loss, dam_acc = dam_net.evaluate(val_ds)
print("Baseline MobileNet Val Acc:", baseline_acc)
print("DAM-Net Val Acc:", dam_acc)

2672/2672 ━━━━━━━━━━━━━━━━━━━━ 132s 49ms/step - accuracy: 0.0061 - loss: 20.4235
2672/2672 ━━━━━━━━━━━━━━━━━━━━ 132s 49ms/step - accuracy: 0.9334 - loss: 0.2617
Baseline MobileNet Val Acc: 0.006128798238933086
DAM-Net Val Acc: 0.9333551526069641
